# Episode 21: Multiple Agents, One Task: Redundancy & Nested Chats

What if you asked three doctors the same question and compared their answers before deciding on treatment? That's the redundant pattern — and it's how you build confidence in high-stakes agent decisions.

You've built and deployed multi-agent systems. These next episodes explore specialized patterns for when your basic toolkit isn't enough.

**In this episode, you'll learn:**
- The redundant pattern: multiple agents independently solve the same task, then an evaluator picks the best
- Nested chats: isolated sub-conversations that keep contexts separate
- When redundancy is worth the extra cost — and when it's wasteful

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Part 1 — The Redundant Pattern

### Theory

The redundant pattern has multiple agents independently work on the **same task**, then an evaluator compares and synthesizes their results.

Think of it as asking 3 experts the same question and picking the best answer — or combining the best parts of each.

```
            ┌─── Agent A (Analytical) ───┐
            │                             │
Task ───────┼─── Agent B (Creative)   ───┼─── Evaluator ─── Final Answer
            │                             │
            └─── Agent C (Practical)  ───┘
```

Each agent works **in isolation** — they don't see each other's responses. This prevents groupthink and ensures diverse perspectives.

### When to Use / When NOT to Use

| Use When | Avoid When |
|----------|------------|
| High-stakes decisions (medical, legal, financial) | Simple factual questions |
| Creative tasks where diversity matters | Time-sensitive responses |
| Validation of critical outputs | Budget-constrained projects |
| You need confidence in the answer | The task has one obvious answer |

**Cost warning:** The redundant pattern uses 3x+ the tokens of a single agent. Each agent processes the full prompt independently, and then the evaluator processes all responses. Use it when quality justifies the cost.

In [ ]:
from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

TASK = """Propose a strategy for reducing customer churn in a SaaS product
that has seen a 15% increase in cancellations over the last quarter."""

In [ ]:
# Create 3 agents with different approaches
with llm_config:
    analytical_agent = ConversableAgent(
        "analytical_expert",
        system_message="""You are a data-driven analyst. Approach every problem with metrics,
        frameworks, and quantitative reasoning. Cite specific KPIs and measurement strategies.
        Be thorough and structured.""",
        human_input_mode="NEVER",
    )

    creative_agent = ConversableAgent(
        "creative_expert",
        system_message="""You are a creative strategist. Think outside the box.
        Propose innovative, unconventional approaches that competitors haven't tried.
        Focus on emotional connection and brand experience.""",
        human_input_mode="NEVER",
    )

    practical_agent = ConversableAgent(
        "practical_expert",
        system_message="""You are a pragmatic operations expert. Focus on what can be
        implemented quickly with existing resources. Prioritize by effort vs impact.
        Be specific about timelines and resource requirements.""",
        human_input_mode="NEVER",
    )

    # The user proxy to drive each conversation
    user = ConversableAgent("user", human_input_mode="NEVER")

In [ ]:
# Run each agent independently on the same task
print("=" * 60)
print("ANALYTICAL AGENT")
print("=" * 60)
result_a = user.initiate_chat(analytical_agent, message=TASK, max_turns=1)

print("\n" + "=" * 60)
print("CREATIVE AGENT")
print("=" * 60)
result_b = user.initiate_chat(creative_agent, message=TASK, max_turns=1)

print("\n" + "=" * 60)
print("PRACTICAL AGENT")
print("=" * 60)
result_c = user.initiate_chat(practical_agent, message=TASK, max_turns=1)

In [ ]:
# Now the evaluator synthesizes all three responses
with llm_config:
    evaluator = ConversableAgent(
        "evaluator",
        system_message="""You are an expert evaluator. You will receive three independent
        responses to the same problem. Your job is to:
        1. Score each response (1-10) on: completeness, feasibility, and originality
        2. Identify the strongest elements from each
        3. Synthesize a final recommendation that combines the best ideas
        Be specific about which ideas you're taking from which expert.""",
        human_input_mode="NEVER",
    )

evaluation_prompt = f"""Evaluate these three independent responses to the same task.

TASK: {TASK}

--- RESPONSE A (Analytical) ---
{result_a.summary}

--- RESPONSE B (Creative) ---
{result_b.summary}

--- RESPONSE C (Practical) ---
{result_c.summary}

Score each, identify strengths, and synthesize the best combined strategy."""

final_result = user.initiate_chat(evaluator, message=evaluation_prompt, max_turns=1)
print("\n" + "=" * 60)
print("FINAL SYNTHESIZED STRATEGY")
print("=" * 60)
print(final_result.summary)

## Part 2 — Nested Chats Deep Dive

### Theory

Nested chats create **isolated sub-conversations** that don't pollute the main conversation thread. Think of them as "side meetings" — agents discuss privately, then report back to the main group.

```
Main Conversation
│
├── Message 1
├── Message 2
├── [Nested Chat triggered]
│   ├── Sub-message A  ← isolated context
│   ├── Sub-message B  ← isolated context
│   └── Sub-message C  ← isolated context
├── Message 3 (includes nested chat summary)
└── Message 4
```

The main conversation only sees the **summary** of the nested chat, not every message within it.

### Use Cases Beyond Redundancy

**1. Sub-team Coordination**

A manager agent spawns a private discussion between specialists:
```python
# Manager detects a technical question → spawns nested chat
# between backend_dev and security_reviewer
# Main conversation gets: "The team recommends approach X"
```

**2. Keeping Contexts Separate**

Prevent one agent's data from confusing another:
```python
# Legal review happens in nested chat → isolated from marketing discussion
# Marketing never sees the legal jargon, only the verdict
```

**3. Parallel Workstreams**

Multiple independent discussions happening simultaneously:
```python
# Nested chat 1: Design team discusses UI
# Nested chat 2: Backend team discusses API
# Results merged in main conversation
```

### The NestedChatTarget Pattern

AG2 provides `NestedChatTarget` for structured nested conversations. Here's the conceptual structure:

```python
from autogen import GroupChat, NestedChatTarget

# Define a nested chat that runs when triggered
nested_config = {
    "chat_queue": [
        {
            "recipient": specialist_agent,
            "message": "Analyze this from your perspective.",
            "max_turns": 2,
        }
    ]
}

# The nested chat runs in isolation, returns a summary
# Main conversation continues with the summary
```

The key insight: nested chats are **composable**. You can nest chats within nested chats, creating arbitrarily deep hierarchies of isolated conversations.

### Comparison: Nested Chat vs Sequential Chat

| Feature | Sequential Chat | Nested Chat |
|---------|----------------|-------------|
| **Context** | Shared — each agent sees prior messages | Isolated — sub-conversation is private |
| **Memory** | Accumulates across all agents | Summarized back to parent |
| **Token usage** | Grows with each step | Resets per nested chat |
| **Use case** | Pipeline processing | Side discussions, validation |
| **Risk** | Context pollution | Information loss in summarization |

**Rule of thumb:** Use sequential chat when agents need to build on each other's work. Use nested chats when agents need to work independently without interference.

## Try It Yourself

Use the redundant pattern for **code review**:

1. Create 3 reviewer agents with different focuses:
   - **Security reviewer** — looks for vulnerabilities
   - **Performance reviewer** — looks for inefficiencies
   - **Readability reviewer** — looks for maintainability issues
2. Give them all the same code snippet to review independently
3. Have an evaluator synthesize the reviews into a final report

This mirrors real-world code review where multiple reviewers catch different issues.

## Reference

- Full NestedChatTarget implementation: Module 4 `4.7_redundant.ipynb` in the workshop materials
- When redundancy improves quality: high variance tasks (creative writing, strategy, design)
- When it's wasteful: low variance tasks (math, lookups, formatting)

---

**What's Next:** In Episode 22, you'll explore advanced reasoning strategies.